In [ ]:
# Imports & Globals
import os, re
import numpy as np
import pandas as pd
import wfdb
from scipy.signal import resample
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils import class_weight

np.random.seed(42)
tf.random.set_seed(42)

DATA_DIR   = 'data/PhysioNet-Leipzig'
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

WIN_S           = 2      # seconds
DOWNSAMPLE_RATE = 200    # Hz
OVERLAP         = 0.0
MAX_WINDOWS     = 1000
AUG_OVERLAP     = 0.5    # for augmentation
AUG_MAX_WINDOWS = 40

In [ ]:
# Utility functions
def parse_age(age_str):
    s = re.sub(r'[^0-9\.]', '', str(age_str))
    parts = s.split('.')
    if len(parts) > 2:
        s = parts[0] + '.' + ''.join(parts[1:])
    try:
        return float(s)
    except:
        return 0.0

NORMAL = {'N','~'}
def extract_windows(rid, base_dir, win_s, overlap, max_windows, dsr, lead2idx):
    path = os.path.join(base_dir, rid)
    rec  = wfdb.rdrecord(path)
    fs   = rec.fs
    sig  = np.zeros((rec.p_signal.shape[0], len(lead2idx)), dtype=np.float32)
    # map only standard leads
    for j, lead in enumerate(rec.sig_name):
        if lead in lead2idx:
            sig[:, lead2idx[lead]] = rec.p_signal[:, j]
    ann = wfdb.rdann(path, 'atr')
    L   = int(win_s * fs)
    step= int(L * (1 - overlap))
    wins, labs = [], []
    for start in range(0, sig.shape[0] - L + 1, step):
        end = start + L
        idxs = np.where((ann.sample >= start)&(ann.sample < end))[0]
        if not len(idxs): continue
        syms = [ann.symbol[i] for i in idxs]
        lbl  = 0 if all(s in NORMAL for s in syms) else 1
        w    = sig[start:end]
        w_ds = resample(w, int(win_s * dsr), axis=0)
        wins.append(w_ds); labs.append(lbl)
    if len(wins) > max_windows:
        sel = np.random.choice(len(wins), max_windows, False)
        wins = [wins[i] for i in sel]
        labs = [labs[i] for i in sel]
    return wins, labs


In [ ]:
# Load pediatric metadata & define 12‑lead mapping
kids = pd.read_csv(os.path.join(DATA_DIR, 'children-subject-info.csv'))
kids['record_id'] = kids['file_name']

STANDARD_LEADS = [
    'I','II','III','aVR','aVL','aVF',
    'V1','V2','V3','V4','V5','V6'
]
lead2idx = {lead: i for i, lead in enumerate(STANDARD_LEADS)}
print('Using leads:', STANDARD_LEADS)

In [ ]:
# Build pediatric dataset
X, y, M, wr = [], [], [], []
for _, r in kids.iterrows():
    wins, labs = extract_windows(
        r['record_id'], DATA_DIR,
        WIN_S, OVERLAP, MAX_WINDOWS,
        DOWNSAMPLE_RATE, lead2idx
    )
    age = parse_age(r['age'])
    sex = 1 if r['gender']=='M' else 0
    for w, lbl in zip(wins, labs):
        X.append(StandardScaler().fit_transform(w))
        y.append(lbl)
        M.append([age, sex])
        wr.append(r['record_id'])

X = np.stack(X)
y = np.array(y)
M = np.array(M)
wr= np.array(wr)

print(f'Total pediatric windows: {len(y)} ({y.sum()} positive)')


In [ ]:
# Pediatric train/test split
gss       = GroupShuffleSplit(test_size=0.2, random_state=42)
child_idx = np.arange(len(y))  # all are children
tr, te    = next(gss.split(X, y, groups=wr))
X_tr, X_te = X[tr], X[te]
y_tr, y_te = y[tr], y[te]
M_tr, M_te = M[tr], M[te]
wr_tr, wr_te = wr[tr], wr[te]
print(f'Train windows: {len(y_tr)}, Test windows: {len(y_te)}')


In [ ]:
# Augmentation on training set
aug_X, aug_y, aug_M = [], [], []
for rid in np.unique(wr_tr):
    wins, labs = extract_windows(
        rid, DATA_DIR,
        WIN_S, AUG_OVERLAP, AUG_MAX_WINDOWS,
        DOWNSAMPLE_RATE, lead2idx
    )
    row = kids[kids['record_id']==rid].iloc[0]
    age = parse_age(row['age'])
    sex = 1 if row['gender']=='M' else 0
    for w, lbl in zip(wins, labs):
        aug_X.append(w)
        aug_y.append(lbl)
        aug_M.append([age, sex])
if aug_X:
    X_tr = np.concatenate([X_tr, np.stack(aug_X)], axis=0)
    y_tr = np.concatenate([y_tr, np.array(aug_y)], axis=0)
    M_tr = np.concatenate([M_tr, np.array(aug_M)], axis=0)

# Gaussian noise
noise = X_tr + np.random.normal(0, 0.01, X_tr.shape)
X_tr = np.concatenate([X_tr, noise], axis=0)
y_tr = np.concatenate([y_tr, y_tr], axis=0)
M_tr = np.concatenate([M_tr, M_tr], axis=0)

print(f'After augmentation: {len(y_tr)} windows')


In [ ]:
# Model builder (HeartCHilD)
def build_model(reg=1e-4, drop=0.3):
    ie = layers.Input(shape=(X_tr.shape[1], X_tr.shape[2]), name='ecg')
    x  = layers.Conv1D(32, 5, activation='relu', kernel_regularizer=l2(reg))(ie)
    x  = layers.MaxPooling1D(2)(x)
    x  = layers.Dropout(drop)(x)
    x  = layers.Conv1D(64, 5, activation='relu', kernel_regularizer=l2(reg))(x)
    x  = layers.GlobalAveragePooling1D()(x)
    im = layers.Input(shape=(2,), name='meta')
    m  = layers.Dense(16, activation='relu', kernel_regularizer=l2(reg))(im)
    c  = layers.concatenate([x, m])
    out= layers.Dense(1, activation='sigmoid', kernel_regularizer=l2(reg))(c)
    return models.Model([ie, im], out)


In [ ]:
# Train & save HeartCHilD
# compute class weights
gcw = class_weight.compute_class_weight('balanced',
                                        classes=np.unique(y_tr),
                                        y=y_tr)
cw  = dict(zip(np.unique(y_tr), gcw))

model = build_model()
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=[tf.keras.metrics.AUC(name='auc')])

es = callbacks.EarlyStopping(patience=3, restore_best_weights=True)

model.fit(x=[X_tr, M_tr],
          y=y_tr,
          validation_data=([X_te, M_te], y_te),
          epochs=30,
          batch_size=64,
          class_weight=cw,
          callbacks=[es],
          verbose=1)

model.save(os.path.join(MODELS_DIR, 'heartchild.keras'))